# Medical Image Segmentation: Episodic Meta-Learning Pipeline
This notebook trains our lightweight in-context adapter (FusionModule + Decoder) across CT and MRI datasets using **episodic meta-learning** to achieve robust zero-shot generalization.

**Key Highlights:**
- **Frozen Backbone**: UniverSeg encoder (77% of model) is completely locked.
- **Trainable Adapter**: 3-layer cross-attention FusionModule + Decoder (23% of model).
- **Uniform Episodic Sampling**: Uniform probability ($p=0.25$) per organ per step prevents class imbalance.
- **Cross-Platform**: Runs natively via Python in Jupyter on Windows, Linux, and Mac.

## 1. Verify GPU and CUDA Environment

In [ ]:
import os
import sys
import torch

# Ensure project root is in path
if os.path.abspath("..") not in sys.path:
    sys.path.insert(0, os.path.abspath(".."))

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")
    print(f"GPU VRAM:        {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("WARNING: CUDA not detected! PyTorch will run on CPU.")

## 2. Verify Processed Datasets (Spleen, Liver, Heart, BrainTumour)

In [ ]:
datasets = ['spleen', 'liver', 'heart', 'braintumour']
for d in datasets:
    p = f"../data/processed/{d}_images.npy" if os.path.exists(f"../data/processed/{d}_images.npy") else f"data/processed/{d}_images.npy"
    print(f"Dataset [{d:12s}]: {'EXISTS' if os.path.exists(p) else 'MISSING'}")

## 3. Run Joint Episodic Meta-Learning Training (30 Epochs)
Trains the 3-layer FusionModule across all 4 datasets with uniform episode task switching.

In [ ]:
!python -u src/train_episodic.py --epochs 30 --episodes_per_epoch 500 --num_layers 3 --lr 1e-3 --num_support 2 --checkpoint models/checkpoints/best_fusion_episodic.pt --results_file logs/episodic_zeroshot_results.txt

## 4. Run Few-Shot Support Context Ablation ($K \in \{1, 2, 4, 8\}$)
Evaluates how segmentation performance scales with the number of support slices provided in the prompt.

In [ ]:
!python -u src/evaluate_shot_ablation.py --checkpoint models/checkpoints/best_fusion_episodic.pt --shots 1 2 4 8 --output_txt logs/shot_ablation_results.txt --output_png logs/shot_ablation_curve.png

## 5. Leave-One-Dataset-Out (LODO) Zero-Shot Matrix
Trains 4 separate episodic models holding out each dataset completely to evaluate true cross-anatomical zero-shot generalization.

In [ ]:
# (A) Held-Out Spleen
!python -u src/train_episodic.py --train_datasets liver heart braintumour --test_dataset spleen --epochs 25 --episodes_per_epoch 400 --checkpoint models/checkpoints/best_fusion_episodic_heldout_spleen.pt --results_file logs/episodic_lodo_spleen.txt

# (B) Held-Out Liver
!python -u src/train_episodic.py --train_datasets spleen heart braintumour --test_dataset liver --epochs 25 --episodes_per_epoch 400 --checkpoint models/checkpoints/best_fusion_episodic_heldout_liver.pt --results_file logs/episodic_lodo_liver.txt

# (C) Held-Out Heart
!python -u src/train_episodic.py --train_datasets spleen liver braintumour --test_dataset heart --epochs 25 --episodes_per_epoch 400 --checkpoint models/checkpoints/best_fusion_episodic_heldout_heart.pt --results_file logs/episodic_lodo_heart.txt

# (D) Held-Out BrainTumour
!python -u src/train_episodic.py --train_datasets spleen liver heart --test_dataset braintumour --epochs 25 --episodes_per_epoch 400 --checkpoint models/checkpoints/best_fusion_episodic_heldout_braintumour.pt --results_file logs/episodic_lodo_braintumour.txt

## 6. View Results Summary

In [ ]:
import os
summary_file = "logs/episodic_zeroshot_results.txt"
if os.path.exists(summary_file):
    with open(summary_file, "r") as f:
        print(f.read())
else:
    print("Results file not found yet. Run Section 3 above first.")